In [ ]:
import os, random, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── 1. CONFIGURATION & PATHS ──────────────────────────────────
SAVE_DIR_MM = Path('/content/drive/MyDrive/Model_results')
SAVE_DIR_MM.mkdir(parents=True, exist_ok=True)

# TODO: Define your local or drive paths
IMAGE_DIR       = Path('path/to/images')
METADATA_CSV    = Path('path/to/metadata.csv')
GROUNDTRUTH_CSV = Path('path/to/groundtruth.csv')

# TODO: Adjust hyperparameters for Multimodal 11-class classification
CFG = dict(
    img_size      = 224,
    batch_size    = ...,
    num_workers   = 2,
    num_epochs    = ...,
    lr            = ...,
    weight_decay  = 1e-4,
    dropout       = ...,
    test_split    = 0.3, # Supporting 70/15/15 split
    seed          = 42,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])

# ── 2. DATA PREPARATION (11-CLASS MULTIMODAL) ─────────────────
meta = pd.read_csv(METADATA_CSV)
gt   = pd.read_csv(GROUNDTRUTH_CSV)

ORIG_CLASS_COLS = ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']

# Map one-hot encoded columns to integer labels (0-10)
gt['class_name'] = gt[ORIG_CLASS_COLS].idxmax(axis=1)
gt['label'] = gt['class_name'].apply(lambda x: ORIG_CLASS_COLS.index(x))

df = meta.merge(gt[['lesion_id', 'label']], on='lesion_id', how='inner')

# TODO: Handle missing metadata values (Imputation)
df['age_approx'] = ...
df['site'] = ...

# TODO: Convert categorical metadata into numerical format (One-Hot Encoding)
df_encoded = ...
meta_cols = [c for c in df_encoded.columns if c.startswith(('site_', 'sex_', 'age_approx'))]
META_DIM = len(meta_cols)

# 3-WAY STRATIFIED SPLIT (70/15/15)
unique_lesions = df_encoded.drop_duplicates(subset='lesion_id')
train_l, temp_l = train_test_split(unique_lesions['lesion_id'], test_size=CFG['test_split'],
                                   stratify=unique_lesions['label'], random_state=CFG['seed'])
val_l, test_l = train_test_split(temp_l, test_size=0.50,
                                 stratify=unique_lesions[unique_lesions['lesion_id'].isin(temp_l)]['label'],
                                 random_state=CFG['seed'])

train_df = df_encoded[df_encoded['lesion_id'].isin(train_l)].reset_index(drop=True)
val_df   = df_encoded[df_encoded['lesion_id'].isin(val_l)].reset_index(drop=True)
test_df  = df_encoded[df_encoded['lesion_id'].isin(test_l)].reset_index(drop=True)

# ── 3. DATASET & TRANSFORMATIONS ─────────────────────────────
class ISICDatasetMM(Dataset):
    def __init__(self, dataframe, image_dir, meta_cols, transform=None):
        self.df, self.image_dir, self.meta_cols, self.transform = dataframe, Path(image_dir), meta_cols, transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = None
        for ext in ('.jpg', '.jpeg', '.png', '.JPG'):
            p = self.image_dir / str(row['lesion_id']) / f"{row['isic_id']}{ext}"
            if p.exists():
                img = Image.open(p).convert('RGB')
                break
        if img is None: img = Image.new('RGB', (CFG['img_size'], CFG['img_size']))
        if self.transform: img = self.transform(img)

        meta_data = torch.tensor(row[self.meta_cols].values.astype(np.float32))
        return img, meta_data, int(row['label'])

# TODO: Define training and evaluation transforms
train_tfm = transforms.Compose([...])
eval_tfm = transforms.Compose([...])

# ── 4. MODEL (LATE FUSION - 11 CLASSES) ──────────────────────
class ISICModelMM11(nn.Module):
    def __init__(self, num_classes=11, meta_dim=META_DIM, dropout=0.4):
        super().__init__()
        # TODO: Select Image Backbone
        self.backbone = timm.create_model('...', pretrained=True, num_classes=0)
        img_dim = self.backbone.num_features

        # TODO: Design Metadata Branch
        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, ...),
            nn.BatchNorm1d(...),
            nn.ReLU(),
            nn.Dropout(dropout / 2)
        )

        # TODO: Design Combined Head
        self.head = nn.Sequential(
            nn.Linear(img_dim + ..., ...),
            nn.BatchNorm1d(...),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(..., num_classes)
        )

    def forward(self, x, m):
        x = self.backbone(x)
        m = self.meta_net(m)
        return self.head(torch.cat((x, m), dim=1))

# ── 5. TRAINING HELPERS ──────────────────────────────────────
def run_epoch_mm_11(model, loader, criterion, optimizer=None, phase='train'):
    model.train() if phase == 'train' else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []

    with torch.set_grad_enabled(phase == 'train'):
        for imgs, metas, labels in loader:
            imgs, metas, labels = imgs.to(DEVICE), metas.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs, metas)
            loss = criterion(logits, labels)

            if phase == 'train':
                optimizer.zero_grad(); loss.backward(); optimizer.step()

            probs = torch.softmax(logits, dim=1)
            running_loss += loss.item() * imgs.size(0)
            correct += (probs.argmax(dim=1) == labels).sum().item()
            total += imgs.size(0)
            all_labels.extend(labels.cpu().numpy()); all_probs.extend(probs.detach().cpu().numpy())

    all_labels_np, all_probs_np = np.array(all_labels), np.array(all_probs)
    present_classes = np.unique(all_labels_np)
    valid_auc_labels = [c for c in present_classes if len(np.unique((all_labels_np == c).astype(int))) > 1]

    auc = roc_auc_score(all_labels_np, all_probs_np, multi_class='ovr', labels=valid_auc_labels) if len(valid_auc_labels) > 1 else np.nan
    return running_loss / total, correct / total, auc, all_labels, all_probs

# ── 6. EXECUTION ──────────────────────────────────────────────

# TODO: Initialize DataLoaders
train_loader = DataLoader(...)
val_loader  = DataLoader(...)
test_loader = DataLoader(...)

model = ISICModelMM11(num_classes=11).to(DEVICE)

# TODO: Initialize Optimizer and Freezing Logic
# for param in model.backbone.parameters(): param.requires_grad = False

optimizer = ...
scheduler = ...
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

history = {'epoch':[], 'tr_auc':[], 'vl_auc':[], 'tr_loss':[], 'vl_loss':[], 'tr_acc':[], 'vl_acc':[]}
best_auc = 0.0

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()

    # TODO: Implement unfreezing/re-initializing optimizer after X epochs
    # if epoch == ...:

    tr_loss, tr_acc, tr_auc, _, _ = run_epoch_mm_11(model, train_loader, criterion, optimizer, 'train')
    vl_loss, vl_acc, vl_auc, _, _ = run_epoch_mm_11(model, val_loader,   criterion, None,      'val')
    scheduler.step()

    for k, v in zip(history.keys(), [epoch, tr_auc, vl_auc, tr_loss, vl_loss, tr_acc, vl_acc]): history[k].append(v)

    # Save best model logic
    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), SAVE_DIR_MM / 'best_mm_model_11class.pth')

# ── 7. FINAL TEST EVALUATION ──────────────────────────────────
print("\n--- Final Evaluation (11-Class Multimodal) ---")
model.load_state_dict(torch.load(..., weights_only=True))
ts_loss, ts_acc, ts_auc, ts_labels, ts_probs = run_epoch_mm_11(model, test_loader, criterion, phase='test')

ts_preds = np.array(ts_probs).argmax(axis=1)
print(classification_report(ts_labels, ts_preds, target_names=ORIG_CLASS_COLS))

# VISUALIZATIONS
# Confusion Matrix
cm = confusion_matrix(ts_labels, ts_preds, labels=list(range(len(ORIG_CLASS_COLS))))
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=ORIG_CLASS_COLS, yticklabels=ORIG_CLASS_COLS)
plt.title('Test Confusion Matrix (11-Class Multimodal)')
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.show()

# Training Curves
epochs = history['epoch']
plt.figure(figsize=(15,5))
plt.subplot(1,3,1); plt.plot(epochs, history['tr_auc'], label='Train'); plt.plot(epochs, history['vl_auc'], label='Val'); plt.title('AUC (OVR)')
plt.subplot(1,3,2); plt.plot(epochs, history['tr_loss'], label='Train'); plt.plot(epochs, history['vl_loss'], label='Val'); plt.title('Loss')
plt.subplot(1,3,3); plt.plot(epochs, history['tr_acc'], label='Train'); plt.plot(epochs, history['vl_acc'], label='Val'); plt.title('Accuracy')
plt.tight_layout(); plt.show()